# 01 · MIDUS 2 data processing

Turns the raw MIDUS 2 Biomarker Project export into the analysis-ready dataset used for latent profile analysis (LPA).

**What it produces**

| Group | Variables |
|---|---|
| LPA indicators (7) | LF-HRV, HF-HRV, RMSSD, heart rate · urinary cortisol · urinary epinephrine, norepinephrine (creatinine-adjusted) |
| Continuous outcomes | trait anxiety, MASQ anxious distress, trait anger, CES-D depression total, count of major health events |
| Binary outcome | `cvd_risk` = any of heart disease, diabetes, stroke, high blood pressure |

**Steps** (all logic lives in `src/midus_stress/`, which is unit-tested):

1. Extract variables and apply **per-column** missing-data rules (`config.py`)
2. Average the two resting baselines (B1, B2) for each HRV measure
3. Derive the depression total and CVD-risk flag, keeping unknowns as missing
4. Listwise deletion, merge biomarkers with outcomes
5. **Box-Cox** transform the raw indicators (heart rate excluded), then z-score

> **Data access.** MIDUS data are distributed by ICPSR under terms that do not allow redistribution, so no data is stored in this repo.
> Download study 29282 from ICPSR, or set `USE_SYNTHETIC = True` to run on generated data in the same format.

## Setup

In [ ]:
import sys, json
from pathlib import Path

# Make the package importable whether this runs from the repo or in Colab.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from midus_stress import cleaning, pipeline, transforms
from midus_stress.config import BIOMARKERS, OUTCOMES, LPA_INDICATORS
from midus_stress.synthetic import make_synthetic

USE_SYNTHETIC = True   # set False and point RAW_PATH at the real export
RAW_PATH = REPO_ROOT / "data" / "raw" / "midus2_agg_data.tsv"
OUT_PATH = REPO_ROOT / "data" / "processed" / "lpa_input.csv"

### Running in Google Colab
Mount Drive and point `RAW_PATH` at your copy of the export, e.g.
```python
from google.colab import drive; drive.mount('/content/drive')
RAW_PATH = Path('/content/drive/MyDrive/MIDUS 2/Data/midus2_agg_data.tsv')
USE_SYNTHETIC = False
```

## 1 · Load the raw export

In [ ]:
raw = make_synthetic(n=1255, seed=0) if USE_SYNTHETIC else pipeline.load_raw(RAW_PATH)
print(f"{len(raw):,} participants, {raw.shape[1]:,} columns")

## 2 · Audit sentinel codes

MIDUS marks missing data with codes such as `8`, `98`, `99998`. Which codes are valid depends on the column: `98` is a real heart rate but a missing-code for a questionnaire score. This table shows every candidate code found and whether the rules in `config.py` remove it. Check anything marked `False` against the ICPSR codebook.

In [ ]:
audit = cleaning.audit_sentinels(raw, BIOMARKERS + OUTCOMES)
audit

## 3 · Run the pipeline

In [ ]:
cfg = pipeline.PipelineConfig(hrv_baseline="mean", transform="boxcox", standardize_outcomes=True)
result = pipeline.run(raw, cfg)
data, report = result.data, result.report

print(json.dumps(report["rows"], indent=2))
print("CVD risk counts:", report["cvd_risk_counts"])

## 4 · Check indicator distributions before and after the transform

In [ ]:
import pandas as pd
before = pd.DataFrame(report["raw_distribution"]).T[["skewness", "kurtosis"]]
after = pd.DataFrame(report["final_distribution"]).T[["skewness", "kurtosis"]]
pd.concat({"raw": before, "Box-Cox + z": after}, axis=1).round(2)

In [ ]:
pd.DataFrame(report["transform_params"]).T.round(3)  # Box-Cox lambda per indicator

## 5 · Save

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(OUT_PATH, index=False)
OUT_PATH.with_suffix(".report.json").write_text(json.dumps(report, indent=2, default=float))
print(f"Saved {len(data):,} rows to {OUT_PATH}")